# Makilala FoS Pipeline — v10

**Run All, top to bottom.** Every number the manuscript reports is recomputed here
and checked against its published value; the last cell prints one block you can
paste back in full.

### What changed from v9

v9 is *correct* — v10 reproduces it, it does not replace it. Two structural
changes only:

1. **No global rebinding.** v9 bound `alpha`, `c`, `scored` … as loose globals and
   later cells reassigned some of them, so by the end of a Run All
   `alpha[scored].min()` came back `0.10` when the scored mask enforces
   `alpha >= 10`. Here the canonical arrays live in a frozen namespace `P`, the
   evaluation mask is the module constant `SCORED`, and `guard()` at the top of
   every downstream cell re-asserts both. Drift becomes an exception, not a
   silent wrong number.
2. **Correct domain for the parameter table.** v9's Table 3 "Footprint" column
   was computed over the full raster extent (21,106,173 px) instead of the
   municipal footprint (13,728,083 px), which is why saturation appeared to vary
   (0.50–0.60) when it is identically 0.60 inside the municipality — the 0.50
   being `estimate_m`'s missing-data fallback on pixels outside it.

Outputs go to `output_maps_v10/`. Nothing in `output_maps_v9/` is touched.

In [ ]:
# ============================================================================
#  §0  CONFIGURATION
#  The only cell that names a path, a CRS or a constant.
# ============================================================================
import os, sys, json, warnings, time
import numpy as np

# Override with the MAKILALA_DIR environment variable if the data lives
# elsewhere (e.g. running this script outside the desktop notebook).
INPUT_DIR  = os.environ.get("MAKILALA_DIR",
                            "/Users/jayroldarcede/Downloads/makilala_data")
OUTPUT_DIR = os.path.join(INPUT_DIR, "output_maps_v10")
os.makedirs(OUTPUT_DIR, exist_ok=True)

FILE_SLOPE = "Slope_PRSTM51N.tif";        FILE_DEPTH = "Soil Depth.tif"
FILE_PGA   = "PGA 2019 EQ Events.tif";    FILE_GAMMA = "Mak_BulkUnitWeight.tif"
FILE_WATER = "Mak_WaterContent.tif";      FILE_PI    = "Mak_PlasticityIndex.tif"
FILE_FINES = "Mak_Fines.tif";             FILE_SAND  = "Mak_Sand.tif"
FILE_LSI   = "LSI_North Cotabato.shp"

REFERENCE_GRID = 'gamma'      # gamma DEFINES the grid; it is never resampled
ALPHA_MIN      = 10           # degrees
GAMMA_W        = 9.81
NODATA         = -9999.0
ALL_TOUCHED    = False        # `observed` is centre-only; poly_id is touched-cell
Z_FILL_M       = 1.2

# CRS. See the v9 §0 note: the Mak_*.tif tags are mutually inconsistent and the
# EPSG:32651 reading is the verified one; the inventory's .prj says ESRI:102457
# but the GPS ground truth says WGS 84, so it is RELABELLED, never reprojected.
GRID_CRS = "EPSG:32651"
LSI_CRS  = "EPSG:32651"
CRS_OVERRIDE = {'slope': None, 'depth': None, 'pga': None}

# Threshold sweep
THRESHOLDS = np.arange(0.40, 2.05, 0.05)
THRESHOLDS_EXT = np.arange(0.05, 2.05, 0.05)   # floor test

# Primary configuration
PRIMARY_C = 'c_plus10'
PRIMARY_Z = 'z_cap_5m'

# ---- published values, for the PASS/FAIL checks ----------------------------
PUBLISHED = dict(
    footprint_px=13_728_083, scored_px=7_104_440, extent_px=21_106_173,
    poly_province=3140, poly_makilala=1703, poly_scored=1682,
    px_makilala=177_589, px_scored=170_547,
    alpha_ge10_px=7_158_603,   # corrected 2026-09-15; the table footnote
                               # (54,163 removed by depth) already implied it
    theta_star=0.90, youden=0.409, auc=0.7503, hit=0.642, far=0.233,
    blocked_auc=0.667, blocked_sd=0.153, blocked_tiles=45, blocked_total=108,
    formb_mean=0.387, formb_frac=0.831, depth_share=0.715,
)

print(f"input : {INPUT_DIR}")
print(f"output: {OUTPUT_DIR}")
print(f"python: {sys.version.split()[0]}   numpy: {np.__version__}")

In [ ]:
# ============================================================================
#  §1  HELPERS
#  Model functions are byte-for-byte the v9 definitions.
# ============================================================================
import rasterio
from rasterio.crs import CRS
from rasterio.warp import reproject, Resampling
from rasterio.features import rasterize
try:
    import geopandas as gpd
    from shapely.geometry import mapping
except ImportError:
    gpd = None; print("!! geopandas missing -- inventory steps will fail")


def estimate_phi(pi, sand, fines):
    phi = np.full_like(pi, 30.0, dtype=np.float64)
    v = (pi > 5) & np.isfinite(pi)
    phi[v] = 35.0 - 0.1 * pi[v]
    no = ~v & np.isfinite(sand)
    phi[no & (sand > 85) & (fines < 10)] = 33.0
    phi[no & (sand > 70) & (fines >= 10) & (fines < 20)] = 31.0
    phi[no & (fines >= 20)] = 28.0
    return np.clip(phi, 20.0, 38.0)


def estimate_c(fines, sand):
    c = np.full_like(fines, 5.0, dtype=np.float64)
    v = np.isfinite(fines) & np.isfinite(sand)
    c[v & (fines < 5)] = 1.0
    c[v & (fines >= 5) & (fines < 12)] = 3.0
    c[v & (fines >= 12) & (fines < 20)] = 6.0
    c[v & (fines >= 20)] = 10.0
    return c


def estimate_m(water):
    '''NB: 0.50 is the MISSING-DATA FALLBACK, not a saturation band.
    The bands are 0.2 / 0.4 / 0.6 / 0.8.'''
    m = np.full_like(water, 0.5, dtype=np.float64)
    v = np.isfinite(water) & (water > 0)
    m[v & (water < 20)] = 0.2
    m[v & (water >= 20) & (water < 35)] = 0.4
    m[v & (water >= 35) & (water < 50)] = 0.6
    m[v & (water >= 50)] = 0.8
    return m


def compute_FoS(c, phi_deg, alpha_deg, m, gamma, z, k_h):
    phi, alpha = np.radians(phi_deg), np.radians(alpha_deg)
    R = c + (gamma - m * GAMMA_W) * z * np.cos(alpha)**2 * np.tan(phi)
    D = gamma * z * np.sin(alpha) * np.cos(alpha) + k_h * gamma * z * np.cos(alpha)**2
    return np.where(D > 0, R / D, np.nan)


def compute_kc(c, phi_deg, alpha_deg, m, gamma, z):
    phi, alpha = np.radians(phi_deg), np.radians(alpha_deg)
    d = gamma * z * np.cos(alpha)**2
    return np.where(d > 0, c/d + (1 - m*GAMMA_W/gamma)*np.tan(phi) - np.tan(alpha), np.nan)


def sensitivity_FormB(fs):
    return np.where(fs > 1, 1.0 - 1.0/fs, np.nan)


def ok(label, got, want, tol, unit=""):
    '''PASS/FAIL against a published value.'''
    good = abs(got - want) <= tol
    flag = "PASS" if good else "**FAIL**"
    gs = f"{got:,}" if isinstance(got, (int, np.integer)) else f"{got:.4f}"
    ws = f"{want:,}" if isinstance(want, (int, np.integer)) else f"{want:.4f}"
    print(f"  [{flag:8s}] {label:<44} {gs:>14}{unit}   published {ws}{unit}")
    CHECKS.append((label, good, gs, ws))
    return good


CHECKS = []
print("helpers defined")

In [ ]:
# ============================================================================
#  §2  LOAD AND REGISTER
#  Layers already on the reference grid PASS THROUGH untouched. Reprojecting
#  them would apply a datum shift to some and not others -- the Mak_* tags are
#  mutually inconsistent, so at most one can be right.
# ============================================================================
t0 = time.time()

def load_raster(path, label=""):
    with rasterio.open(path) as src:
        d = src.read(1).astype(np.float64)
        p = src.profile.copy()
        if src.nodata is not None:
            d[d == src.nodata] = np.nan
        v = np.isfinite(d) & (d != 0)
        print(f"  {label:<7}: {str(d.shape):>14} @ {src.res[0]:>8.4g}"
              f"  {str(src.crs):<14} valid={int(v.sum()):>11,}")
        return d, p


FILES = {'slope': FILE_SLOPE, 'depth': FILE_DEPTH, 'pga': FILE_PGA,
         'gamma': FILE_GAMMA, 'water': FILE_WATER, 'pi': FILE_PI,
         'fines': FILE_FINES, 'sand': FILE_SAND}

rasters, profiles = {}, {}
for key, fn in FILES.items():
    _path = os.path.join(INPUT_DIR, fn)
    if not os.path.exists(_path):
        raise FileNotFoundError(f"{key}: {_path}")
    rasters[key], profiles[key] = load_raster(_path, key)

ref_shape     = rasters[REFERENCE_GRID].shape
ref_transform = profiles[REFERENCE_GRID]['transform']
ref_crs       = CRS.from_string(GRID_CRS)
ref_profile   = {**profiles[REFERENCE_GRID], 'crs': ref_crs}

print(f"\nreference grid: {ref_shape[0]} x {ref_shape[1]} @ {ref_transform.a:g} m"
      f"  = {ref_shape[0]*ref_shape[1]:,} cells")
print(f"  file tag : {profiles[REFERENCE_GRID].get('crs')}")
print(f"  GRID_CRS : {ref_crs}  <- used\n")

aligned = {}
for key, data in rasters.items():
    same = (data.shape == ref_shape
            and profiles[key]['transform'].almost_equals(ref_transform, precision=1e-6))
    if same:
        aligned[key] = data
        print(f"  = {key:<7}: on the reference grid, passed through untouched")
        continue
    src_crs = CRS_OVERRIDE.get(key) or profiles[key]['crs']
    if src_crs is None:
        raise RuntimeError(f"{key}: no CRS and no override")
    dst = np.full(ref_shape, np.nan, dtype=np.float64)
    reproject(source=data, destination=dst,
              src_transform=profiles[key]['transform'], src_crs=CRS.from_user_input(src_crs),
              dst_transform=ref_transform, dst_crs=ref_crs,
              resampling=Resampling.bilinear, src_nodata=np.nan, dst_nodata=np.nan)
    aligned[key] = dst
    print(f"  > {key:<7}: reprojected from {src_crs}, coverage "
          f"{100*np.isfinite(dst).mean():.1f}% of the grid")

del rasters
print(f"\nregistration complete in {time.time()-t0:.1f}s")

In [ ]:
# ============================================================================
#  §3  DERIVED PARAMETERS  ->  frozen namespace P
#
#  Everything downstream reads P.<name>. Nothing rebinds a bare `alpha`, `c`
#  or `m` at module level, which is the v9 failure this version exists to stop.
# ============================================================================
from types import SimpleNamespace

_alpha = np.clip(aligned['slope'], 0.1, 80.0)
_gamma = np.clip(aligned['gamma'], GAMMA_W + 0.1, 30.0)
_phi   = estimate_phi(aligned['pi'], aligned['sand'], aligned['fines'])
_c     = estimate_c(aligned['fines'], aligned['sand'])
_m     = estimate_m(aligned['water'])
_z_raw = aligned['depth']

_pga = aligned['pga']
_pmax = np.nanmax(_pga[np.isfinite(_pga) & (_pga > 0)])
_pga_g = _pga/981.0 if _pmax > 100 else (_pga/9.81 if _pmax > 2 else _pga)
_k_h = np.clip(0.5 * _pga_g, 0.0, 1.0)

# FOOTPRINT: where the interpolated geotechnical rasters carry values.
# v9 uses isfinite(gamma) & isfinite(alpha); water gives the identical set.
FOOTPRINT = np.isfinite(_gamma) & np.isfinite(_alpha)

# gap-filled depth, for isolating the coverage effect
_z_missing = FOOTPRINT & (~np.isfinite(_z_raw) | (_z_raw <= 0))

# Only what is read downstream is retained. At 169 MB per float64 array on a
# 21.1 M-cell grid, keeping the full `aligned` dict as well costs 1.3 GB for
# nothing, and the sweep in §7 needs that headroom.
P = SimpleNamespace(alpha=_alpha, gamma=_gamma, phi=_phi, c=_c, m=_m,
                    z_raw=_z_raw, z_missing=_z_missing,
                    k_h=_k_h, pga_g=_pga_g, water=aligned['water'])
for _a in vars(P).values():
    if isinstance(_a, np.ndarray):
        _a.flags.writeable = False          # make accidental mutation raise

del aligned, _alpha, _gamma, _phi, _c, _m, _z_raw, _k_h, _pga_g
import gc; gc.collect()

_PIN = {'scored_sig': None}
_FINGERPRINT = {k: (float(np.nansum(v)), v.shape)
                for k, v in vars(P).items() if isinstance(v, np.ndarray)}


def guard():
    '''Re-assert that P and SCORED are intact. Call at the top of every cell.'''
    for k, (s, sh) in _FINGERPRINT.items():
        a = getattr(P, k)
        if a.shape != sh or not np.isclose(float(np.nansum(a)), s, rtol=0, atol=1e-6):
            raise RuntimeError(f"P.{k} has been mutated since §3 -- rerun from the top")
    if 'SCORED' in globals():
        n = int(SCORED.sum())
        # Count alone is not enough: the v9 failure was a mask with the RIGHT
        # cardinality and the WRONG membership. Two cheap spatial moments (row
        # and column profiles) pin the actual footprint of the mask, not just
        # its size.
        _rs = SCORED.sum(axis=1); _cs = SCORED.sum(axis=0)
        sig = (n,
               int((_rs * np.arange(_rs.size)).sum()),
               int((_cs * np.arange(_cs.size)).sum()))
        if _PIN.get('scored_sig') is None:
            _PIN['scored_sig'] = sig
        elif sig != _PIN['scored_sig']:
            was, now = _PIN['scored_sig'], sig
            what = ("count" if was[0] != now[0] else "membership (same count)")
            raise RuntimeError(
                f"SCORED has changed -- {what}. Was {was}, now {now}. "
                f"Something rebound or mutated it; rerun from the top.")
        # NaN-safe: a plain .min() over a mask that includes nodata pixels
        # returns nan, and `nan < ALPHA_MIN` is False -- the check would pass
        # silently on exactly the corruption it is meant to catch.
        _av = P.alpha[SCORED]
        _nn = int(np.count_nonzero(~np.isfinite(_av)))
        if _nn:
            raise RuntimeError(f"SCORED selects {_nn:,} px where alpha is not "
                               f"finite -- it is not the evaluation domain")
        am = float(np.nanmin(_av))
        if am < ALPHA_MIN - 1e-9:
            raise RuntimeError(f"SCORED contains alpha={am:.2f} deg < {ALPHA_MIN} "
                               f"-- it is not the evaluation domain any more")
    return True


print(f"footprint (isfinite gamma & alpha): {int(FOOTPRINT.sum()):,}")
_alt = int((np.isfinite(P.water) & np.isfinite(P.alpha)).sum())
print(f"footprint (isfinite water & alpha): {_alt:,}   {'(identical)' if _alt==int(FOOTPRINT.sum()) else '(DIFFERS)'}")
print(f"P frozen with {len(_FINGERPRINT)} arrays; guard() armed")

In [ ]:
# ============================================================================
#  §4  INVENTORY
#  The shapefile's .prj says ESRI:102457. It is RELABELLED to EPSG:32651, not
#  reprojected -- see the §0 note. poly_id is touched-cell; `observed` is
#  centre-only. That asymmetry is deliberate and is reported in Table 2.
# ============================================================================
guard()

lsi = gpd.read_file(os.path.join(INPUT_DIR, FILE_LSI))
print(f"inventory .prj declares : {lsi.crs}")
lsi = lsi.set_crs(CRS.from_string(LSI_CRS), allow_override=True)
print(f"relabelled to           : {lsi.crs}  (NOT reprojected)")

if 'ESI_local' not in lsi.columns:
    _cand = [c for c in lsi.columns if 'esi' in c.lower()]
    if _cand:
        lsi['ESI_local'] = lsi[_cand[0]]
        print(f"ESI column: using '{_cand[0]}'")
    else:
        lsi['ESI_local'] = np.nan
        print("!! no ESI column found; ESI stratification will be skipped")
lsi['area'] = lsi.geometry.area

poly_id = rasterize(
    [(mapping(g), i+1) for i, g in enumerate(lsi.geometry) if g is not None],
    out_shape=ref_shape, transform=ref_transform, fill=0, dtype=np.int32,
    all_touched=True)

_ncov = np.bincount(poly_id.ravel(), weights=FOOTPRINT.ravel(), minlength=len(lsi)+1)[1:]
_nany = np.bincount(poly_id.ravel(), minlength=len(lsi)+1)[1:]
in_mak  = _ncov > 0
lsi_mak = lsi.loc[in_mak].copy()

OBSERVED = rasterize([(mapping(g), 1) for g in lsi_mak.geometry if g is not None],
                     out_shape=ref_shape, transform=ref_transform, fill=0,
                     dtype=np.uint8, all_touched=ALL_TOUCHED).astype(bool)
OBSERVED.flags.writeable = False

ESI_RASTERS = {}
for _e in sorted(x for x in lsi_mak['ESI_local'].dropna().unique()):
    _sub = lsi_mak[lsi_mak['ESI_local'] == _e]
    ESI_RASTERS[int(_e)] = rasterize(
        [(mapping(g), 1) for g in _sub.geometry if g is not None],
        out_shape=ref_shape, transform=ref_transform, fill=0,
        dtype=np.uint8, all_touched=ALL_TOUCHED).astype(bool)

print()
ok("inventory: province-wide polygons", len(lsi), PUBLISHED['poly_province'], 0)
ok("inventory: within Makilala footprint", len(lsi_mak), PUBLISHED['poly_makilala'], 0)
ok("positive px within footprint", int((OBSERVED & FOOTPRINT).sum()),
   PUBLISHED['px_makilala'], 0)
print(f"  rasterised over the whole grid : {int(OBSERVED.sum()):,}"
      f"  (+{int(OBSERVED.sum())-int((OBSERVED & FOOTPRINT).sum()):,} outside the footprint:")
print( "   a polygon selected because it TOUCHES the footprint can still have")
print( "   centre-pixels outside it, and Table 2 counts those out.)")
print(f"  outside raster extent    : {int((_nany == 0).sum()):,}")
print(f"  touching but uncovered   : {int((~in_mak).sum() - (_nany == 0).sum()):,}")
print(f"  ESI levels               : {sorted(ESI_RASTERS)}")

In [ ]:
# ============================================================================
#  §5  DOMAIN CASCADE  ->  Table 2, and the definition of SCORED
# ============================================================================
guard()

PIXEL_M2 = abs(ref_transform.a * ref_transform.e)

_z_primary = np.clip(P.z_raw, 0.1, 5.0)     # primary cap; §6 sweeps the others
_valid = (np.isfinite(P.alpha) & (P.alpha >= ALPHA_MIN)
          & np.isfinite(P.z_raw) & (P.z_raw > 0)
          & np.isfinite(P.gamma) & (P.gamma > GAMMA_W)
          & np.isfinite(P.phi)   & (P.phi > 0)
          & np.isfinite(P.c)     & (P.c >= 0)
          & np.isfinite(P.m)     & (P.m >= 0) & (P.m <= 1))

_fos_primary = np.full(ref_shape, np.nan)
_fos_primary[_valid] = compute_FoS(
    (P.c + 10.0)[_valid], P.phi[_valid], P.alpha[_valid], P.m[_valid],
    P.gamma[_valid], _z_primary[_valid], P.k_h[_valid])

SCORED = _valid & np.isfinite(_fos_primary)
SCORED.flags.writeable = False
guard()

n_fp   = int(FOOTPRINT.sum())
n_a10  = int((FOOTPRINT & (P.alpha >= ALPHA_MIN)).sum())
n_sc   = int(SCORED.sum())

print("Evaluation domain")
print(f"  municipal footprint              {n_fp:>12,}   {n_fp*PIXEL_M2/1e6:>7.1f} km2")
print(f"  + alpha >= {ALPHA_MIN} deg                 {n_a10:>12,}   {n_a10*PIXEL_M2/1e6:>7.1f} km2")
print(f"  + measured soil depth            {n_sc:>12,}   {n_sc*PIXEL_M2/1e6:>7.1f} km2")
print(f"    alpha filter removes           {n_fp-n_a10:>12,}")
print(f"    depth adds a further           {n_a10-n_sc:>12,}   "
      f"({100*(n_a10-n_sc)/max(n_a10,1):.1f}% of what remains)")
print(f"  SoilGrids covers                 "
      f"{100*np.sum(FOOTPRINT & np.isfinite(P.z_raw) & (P.z_raw>0))/n_fp:>11.1f}% of the footprint")

_has = np.bincount(poly_id.ravel(), weights=SCORED.ravel(), minlength=len(lsi)+1)[1:]
n_poly_scored = int(((_has > 0) & in_mak).sum())

print()
ok("footprint px", n_fp, PUBLISHED['footprint_px'], 0)
ok("alpha >= 10 px", n_a10, PUBLISHED['alpha_ge10_px'], 0)
ok("scored px", n_sc, PUBLISHED['scored_px'], 0)
ok("polygons with >=1 scored px", n_poly_scored, PUBLISHED['poly_scored'], 0)
ok("positive px in scored domain", int((SCORED & OBSERVED).sum()), PUBLISHED['px_scored'], 0)

In [ ]:
# ============================================================================
#  §6  PARAMETER STATISTICS  ->  Table 3
#  BOTH columns over their stated domains. v9's footprint column was computed
#  over the full raster extent, which is what put 0.50 in the saturation row.
# ============================================================================
guard()

_rows = [("Slope angle alpha (deg)",        P.alpha,              P.alpha),
         ("Soil depth z (m)",               P.z_raw,              np.clip(P.z_raw, 0.1, 5.0)),
         ("Bulk unit weight gamma (kN/m3)", P.gamma,              P.gamma),
         ("Friction angle phi' (deg)",      P.phi,                P.phi),
         ("Cohesion c' (kPa)",              P.c,                  P.c + 10.0),
         ("Saturation ratio m",             P.m,                  P.m),
         ("Seismic coefficient k_h",        P.k_h,                P.k_h)]

print(f"{'Parameter':<32}{'--- Footprint ---':>30}{'--- Scored domain ---':>32}")
print(f"{'':<32}{'min':>9}{'max':>10}{'mean':>11}{'min':>10}{'max':>10}{'mean':>11}")
print("-" * 94)
TABLE3 = {}
for name, a_fp, a_sc in _rows:
    f = a_fp[FOOTPRINT & np.isfinite(a_fp)]
    s = a_sc[SCORED    & np.isfinite(a_sc)]
    TABLE3[name] = (f.min(), f.max(), f.mean(), s.min(), s.max(), s.mean())
    print(f"{name:<32}{f.min():>9.2f}{f.max():>10.2f}{f.mean():>11.3f}"
          f"{s.min():>10.2f}{s.max():>10.2f}{s.mean():>11.3f}")

print("\nFor comparison, the FULL RASTER EXTENT -- the domain v9's footprint")
print("column was mistakenly computed over:")
for name, a_fp, _ in _rows:
    e = a_fp[np.isfinite(a_fp)]
    print(f"  {name:<32}{e.min():>9.2f}{e.max():>10.2f}{e.mean():>11.3f}")

_w = P.water[FOOTPRINT]
print(f"\nWater content over the footprint: {_w.min():.4f} - {_w.max():.4f} %")
print(f"  margin to the 35% band edge    : {_w.min()-35.0:.4f} percentage points")
for _d in (1.0, 2.0, 3.0):
    _n = int((_w < 35.0+_d).sum())
    print(f"  px within {_d:.0f} pp of the edge      : {_n:>10,}  ({100*_n/len(_w):.3f}%)")
_mb = {v: int((FOOTPRINT & (P.m == v)).sum()) for v in (0.2, 0.4, 0.5, 0.6, 0.8)}
print(f"  saturation bands on footprint  : "
      + "  ".join(f"m={k}:{v:,}" for k, v in _mb.items() if v))

In [ ]:
# ============================================================================
#  §7  COHESION x DEPTH x THRESHOLD SWEEP  ->  Table 1
# ============================================================================
guard()
t0 = time.time()

Z_SCENARIOS = {'z_original': 20.0, 'z_cap_5m': 5.0, 'z_cap_3m': 3.0}
C_SCENARIOS = {'c_baseline': 0.0, 'c_plus5': 5.0, 'c_times2': None, 'c_plus10': 10.0}


def sweep_config(c_arr, z_arr, keep=False):
    v = (np.isfinite(P.alpha) & (P.alpha >= ALPHA_MIN)
         & np.isfinite(z_arr) & (z_arr > 0)
         & np.isfinite(P.gamma) & (P.gamma > GAMMA_W)
         & np.isfinite(P.phi) & (P.phi > 0)
         & np.isfinite(c_arr) & (c_arr >= 0)
         & np.isfinite(P.m) & (P.m >= 0) & (P.m <= 1))

    fos = np.full(ref_shape, np.nan)
    fos[v] = compute_FoS(c_arr[v], P.phi[v], P.alpha[v], P.m[v],
                         P.gamma[v], z_arr[v], P.k_h[v])
    if keep:
        fos_static = np.full(ref_shape, np.nan)
        fos_static[v] = compute_FoS(c_arr[v], P.phi[v], P.alpha[v], P.m[v],
                                    P.gamma[v], z_arr[v], 0.0)
        kc = np.full(ref_shape, np.nan)
        kc[v] = compute_kc(c_arr[v], P.phi[v], P.alpha[v], P.m[v], P.gamma[v], z_arr[v])
    else:
        fos_static = kc = None

    sc = v & np.isfinite(fos)
    pos = np.sort(fos[sc & OBSERVED]); neg = np.sort(fos[sc & ~OBSERVED])
    npos, nneg = len(pos), len(neg)

    lt = np.searchsorted(neg, pos, 'left'); le = np.searchsorted(neg, pos, 'right')
    auc = ((nneg - le).sum() + 0.5*(le - lt).sum()) / (npos * nneg)

    hr  = np.searchsorted(pos, THRESHOLDS, 'left') / npos
    far = np.searchsorted(neg, THRESHOLDS, 'left') / nneg
    J = hr - far
    i = int(np.nanargmax(J))

    out = dict(auc=auc, hr=hr, far=far, J=J,
                theta=float(THRESHOLDS[i]), J_best=float(J[i]),
                hit=float(hr[i]), far_best=float(far[i]),
                censored=(i == 0 or i == len(THRESHOLDS)-1),
                n_scored=int(sc.sum()),
                c_mean=float(np.nanmean(c_arr[v])), z_mean=float(np.nanmean(z_arr[v])),
                fos_mean=float(np.nanmean(fos[sc])))
    if keep:
        out.update(fos=fos, fos_static=fos_static, kc=kc, scored=sc, pos=pos, neg=neg)
    else:
        del fos, v, sc, pos, neg
    return out


GRID = {}
for cn, coff in C_SCENARIOS.items():
    c_arr = P.c * 2.0 if coff is None else P.c + coff
    for zn, zcap in Z_SCENARIOS.items():
        # full rasters retained for the primary configuration only; the other
        # eleven contribute summary statistics and are then freed.
        GRID[(cn, zn)] = sweep_config(c_arr, np.clip(P.z_raw, 0.1, zcap),
                                      keep=(cn == PRIMARY_C and zn == PRIMARY_Z))
    del c_arr
import gc; gc.collect()

print(f"{'cohesion':<12}{'z-cap':<12}{'c bar':>7}{'z bar':>7}{'FoS bar':>9}"
      f"{'theta*':>8}{'J':>8}{'hit':>8}{'FAR':>8}{'AUC':>9}  flag")
print("-" * 98)
for (cn, zn), r in GRID.items():
    print(f"{cn:<12}{zn:<12}{r['c_mean']:>7.2f}{r['z_mean']:>7.2f}{r['fos_mean']:>9.3f}"
          f"{r['theta']:>8.2f}{r['J_best']:>8.3f}{100*r['hit']:>7.1f}%"
          f"{100*r['far_best']:>7.1f}%{r['auc']:>9.4f}  "
          f"{'CENSORED' if r['censored'] else ''}")

REF = GRID[(PRIMARY_C, PRIMARY_Z)]
_th = [r['theta'] for r in GRID.values()]
_J  = [r['J_best'] for r in GRID.values()]
_au = [r['auc'] for r in GRID.values()]
print(f"\ntheta* range {min(_th):.2f}-{max(_th):.2f}   J range {min(_J):.3f}-{max(_J):.3f}"
      f"   AUC range {min(_au):.4f}-{max(_au):.4f} (spread {max(_au)-min(_au):.4f})")
print(f"censored configurations: {sum(r['censored'] for r in GRID.values())} of {len(GRID)}")
print(f"\nsweep complete in {time.time()-t0:.1f}s")

print()
ok("primary theta*", REF['theta'], PUBLISHED['theta_star'], 0.001)
ok("primary Youden J", REF['J_best'], PUBLISHED['youden'], 0.001)
ok("primary AUC", REF['auc'], PUBLISHED['auc'], 0.0002)
ok("primary hit rate", REF['hit'], PUBLISHED['hit'], 0.002)
ok("primary false alarm rate", REF['far_best'], PUBLISHED['far'], 0.002)

In [ ]:
# ============================================================================
#  §8  PIXEL VALIDATION, EXACT OPTIMUM AND FLOOR TEST
#  J(theta) is a STEP function: it changes only where theta crosses an
#  empirical FoS value, so it is not differentiable and is maximised on an
#  interval. There is no stationarity condition to solve; Eq. (8) is evaluated
#  by enumeration, and the exact optimum is computable in one pass.
# ============================================================================
guard()

pos, neg = REF['pos'], REF['neg']
npos, nneg = len(pos), len(neg)
i = int(np.nanargmax(REF['J']))
th = REF['theta']

# Named N_TP/N_FP rather than tp/fp: a later cell using `fp` for a file path
# would otherwise clobber the count, which is precisely the class of bug this
# notebook exists to prevent.
N_TP = int(np.searchsorted(pos, th, 'left')); N_FN = npos - N_TP
N_FP = int(np.searchsorted(neg, th, 'left')); N_TN = nneg - N_FP
prec = N_TP/(N_TP + N_FP)
FLAGGED_FRAC = 100*(N_TP + N_FP)/(npos + nneg)
print(f"Confusion matrix at theta* = {th:.2f}")
print(f"           flagged      not flagged")
print(f"  slide  {N_TP:>10,}   {N_FN:>13,}   hit rate {N_TP/npos:6.1%}")
print(f"  stable {N_FP:>10,}   {N_TN:>13,}   FAR      {N_FP/nneg:6.1%}")
print(f"  precision {prec:.1%}   -- so {1-prec:.1%} of flagged pixels are false,")
print(f"  against a false ALARM rate of {N_FP/nneg:.1%}: the two differ because")
print(f"  landslide pixels are only {npos/(npos+nneg):.1%} of the scored domain.")
print(f"  flagged fraction of domain: {FLAGGED_FRAC:.1f}%")

# ---- floor test ------------------------------------------------------------
hrE = np.searchsorted(pos, THRESHOLDS_EXT, 'left')/npos
farE = np.searchsorted(neg, THRESHOLDS_EXT, 'left')/nneg
JE = hrE - farE
lo = THRESHOLDS_EXT <= 0.40
FLOOR_MONOTONE = bool(np.all(np.diff(JE[lo]) >= -1e-12))
print(f"\nFloor test on [0.05, 2.00]")
print(f"  theta* unchanged at {THRESHOLDS_EXT[int(np.argmax(JE))]:.2f};"
      f"  max J strictly below 0.40 = {JE[THRESHOLDS_EXT < 0.40].max():.6f}")
print(f"  J monotone increasing on [0.05, 0.40]: {FLOOR_MONOTONE}"
      f"  ->  the 0.40 floor is {'NOT binding' if FLOOR_MONOTONE else 'POSSIBLY BINDING'}")

# ---- exact optimum ---------------------------------------------------------
# Deliberately verbose names: these are read again in the summary cell, and a
# one-letter name here is exactly what a later loop variable would clobber.
CAND_FOS = np.unique(np.concatenate([pos, neg]))
Jl = np.searchsorted(pos, CAND_FOS, 'left')/npos - np.searchsorted(neg, CAND_FOS, 'left')/nneg
Jr = np.searchsorted(pos, CAND_FOS, 'right')/npos - np.searchsorted(neg, CAND_FOS, 'right')/nneg
Jall = np.maximum(Jl, Jr)
K_EXACT = int(np.argmax(Jall)); J_EXACT = float(Jall[K_EXACT])
NEAR_OPT = CAND_FOS[Jall >= J_EXACT - 1e-3]
print(f"\nExact optimum over all real theta ({len(CAND_FOS):,} distinct FoS values)")
print(f"  exact sup J = {J_EXACT:.6f} just above FoS = {CAND_FOS[K_EXACT]:.6f}")
print(f"  grid  max J = {REF['J_best']:.6f} at theta = {th:.2f}")
print(f"  cost of the 0.05 grid = {J_EXACT-REF['J_best']:.6f}"
      f"  ({100*(J_EXACT-REF['J_best'])/J_EXACT:.4f}% of J)")
print(f"  within 1e-3 of the max: FoS {NEAR_OPT.min():.4f} - {NEAR_OPT.max():.4f}"
      f"  (width {NEAR_OPT.max()-NEAR_OPT.min():.4f})")
for d in (-2, -1, 1, 2):
    if 0 <= i+d < len(THRESHOLDS):
        print(f"    theta={THRESHOLDS[i+d]:.2f}  J={REF['J'][i+d]:.6f}"
              f"  ({REF['J'][i+d]-REF['J_best']:+.6f})")

# ---- Form B ----------------------------------------------------------------
fsB = REF['fos_static']
FORMB_DEFINED = SCORED & np.isfinite(fsB) & (fsB > 1)
FORMB_MAP = sensitivity_FormB(fsB)
print(f"\nForm B bound |S|max = 1 - 1/FoS_static")
print(f"  defined on {int(FORMB_DEFINED.sum()):,} px = "
      f"{100*FORMB_DEFINED.sum()/SCORED.sum():.1f}% of scored")
print(f"  mean over that set: {np.nanmean(FORMB_MAP[FORMB_DEFINED]):.4f}")
ok("Form B mean", float(np.nanmean(FORMB_MAP[FORMB_DEFINED])), PUBLISHED['formb_mean'], 0.002)
ok("Form B defined fraction", float(FORMB_DEFINED.sum()/SCORED.sum()),
   PUBLISHED['formb_frac'], 0.003)

In [ ]:
# ============================================================================
#  §9  POLYGON-LEVEL AND ESI-STRATIFIED DETECTION
# ============================================================================
guard()

flagged = SCORED & (REF['fos'] < REF['theta'])
pid_scored = np.where(SCORED, poly_id, 0)
n_tot = np.bincount(pid_scored.ravel(), minlength=len(lsi)+1)[1:]
n_hit = np.bincount(pid_scored.ravel(), weights=flagged.ravel(), minlength=len(lsi)+1)[1:]
has = (n_tot > 0) & in_mak
frac = np.zeros(len(lsi)); frac[has] = n_hit[has]/n_tot[has]

print(f"Polygons with at least one scored pixel: {int(has.sum()):,}")
print(f"{'detected fraction >=':<24}{'polygons':>10}{'share':>9}")
for t in (0.01, 0.10, 0.25, 0.50, 0.75, 1.00):
    n = int((frac[has] >= t).sum())
    print(f"  {t:>5.0%}{'':<17}{n:>10,}{n/has.sum():>9.1%}")
print(f"  mean detected fraction : {frac[has].mean():.1%}")

if ESI_RASTERS:
    print(f"\n{'ESI':<6}{'polygons':>10}{'scored px':>12}{'hit rate':>11}"
          f"{'95% CI (polygon-level)':>26}")
    for e, ras in sorted(ESI_RASTERS.items()):
        sel = has & np.array([ (lsi['ESI_local'].iloc[j] == e) for j in range(len(lsi)) ])
        npoly = int(sel.sum())
        if not npoly: continue
        px = SCORED & ras
        hrate = float(flagged[px].sum()/max(px.sum(), 1))
        p = frac[sel].mean()
        se = (frac[sel].std(ddof=1)/np.sqrt(npoly)) if npoly > 1 else 0.0
        print(f"{e:<6}{npoly:>10,}{int(px.sum()):>12,}{hrate:>11.1%}"
              f"{p-1.96*se:>13.1%}{p+1.96*se:>13.1%}")
    print("  CIs are on polygon counts, not pixels: pixels within a polygon are")
    print("  not independent observations.")

In [ ]:
# ============================================================================
#  §10  SPATIALLY BLOCKED VALIDATION  ->  Section 6.4
#  2 km tiles = 400 x 400 px at 5 m. Within a tile the regional contrast is
#  absent, so the ordering has to be earned slope by slope.
# ============================================================================
guard()

TILE_PX   = int(round(2000 / abs(ref_transform.a)))
MIN_BLOCK = 200

fos = REF['fos']
rows = range(0, ref_shape[0], TILE_PX)
cols = range(0, ref_shape[1], TILE_PX)

aucs, own_theta, n_tiles_total, n_scored_tiles = [], [], 0, 0
pos_ct, neg_ct, npos_t, nneg_t = [], [], [], []
for r0 in rows:
    for c0 in cols:
        sl = (slice(r0, r0+TILE_PX), slice(c0, c0+TILE_PX))
        s = SCORED[sl]
        if not s.any():
            continue
        n_tiles_total += 1
        f = fos[sl]; o = OBSERVED[sl]
        p = np.sort(f[s & o]); q = np.sort(f[s & ~o])
        if len(p) < MIN_BLOCK or len(q) < MIN_BLOCK:
            continue
        n_scored_tiles += 1
        lt = np.searchsorted(q, p, 'left'); le = np.searchsorted(q, p, 'right')
        aucs.append(((len(q)-le).sum() + 0.5*(le-lt).sum())/(len(p)*len(q)))
        # Per-threshold counts are kept so the leave-one-out pools below can be
        # formed by SUBTRACTION, which is exact and costs O(tiles x thresholds)
        # instead of rescanning the rasters 45 times.
        pc = np.searchsorted(p, THRESHOLDS, 'left')
        qc = np.searchsorted(q, THRESHOLDS, 'left')
        pos_ct.append(pc); neg_ct.append(qc)
        npos_t.append(len(p)); nneg_t.append(len(q))
        own_theta.append(float(THRESHOLDS[int(np.argmax(pc/len(p) - qc/len(q)))]))

aucs = np.array(aucs); own_theta = np.array(own_theta)
pos_ct = np.array(pos_ct); neg_ct = np.array(neg_ct)
npos_t = np.array(npos_t); nneg_t = np.array(nneg_t)

# ---- TRUE leave-one-block-out threshold selection --------------------------
# For each scored tile, select theta* on the OTHER tiles and apply it to the
# held-out one. This is what Section 6.4 reports. Computing each tile's own
# optimum instead (the `own_theta` above) answers a different question -- how
# much the local optima scatter -- and must not be compared with that claim.
# "the training tiles" means every tile except the held-out one -- i.e. ALL
# scored pixels outside it, not merely the other tiles that happen to qualify
# for within-block AUC. Using the narrower pool is a real but different
# experiment, reported below as a sensitivity.
GP = np.searchsorted(pos, THRESHOLDS, 'left')      # whole scored domain
GQ = np.searchsorted(neg, THRESHOLDS, 'left')
lobo_theta = np.empty(n_scored_tiles)
for t in range(n_scored_tiles):
    HR  = (GP - pos_ct[t]) / (len(pos) - npos_t[t])
    FAR = (GQ - neg_ct[t]) / (len(neg) - nneg_t[t])
    lobo_theta[t] = THRESHOLDS[int(np.argmax(HR - FAR))]

# sensitivity: restrict the training pool to the other qualifying tiles only
tot_pc, tot_qc = pos_ct.sum(axis=0), neg_ct.sum(axis=0)
tot_np, tot_nq = npos_t.sum(), nneg_t.sum()
lobo_narrow = np.empty(n_scored_tiles)
for t in range(n_scored_tiles):
    HR  = (tot_pc - pos_ct[t]) / (tot_np - npos_t[t])
    FAR = (tot_qc - neg_ct[t]) / (tot_nq - nneg_t[t])
    lobo_narrow[t] = THRESHOLDS[int(np.argmax(HR - FAR))]
print(f"tile size            : {TILE_PX} px = {TILE_PX*abs(ref_transform.a)/1000:.1f} km")
print(f"tiles intersecting   : {n_tiles_total}")
print(f"tiles with >= {MIN_BLOCK} of each class: {n_scored_tiles}")
print(f"\nwithin-block AUC     : {aucs.mean():.3f} +/- {aucs.std(ddof=1):.3f}"
      f"   range {aucs.min():.3f}-{aucs.max():.3f}")
print(f"pooled AUC           : {REF['auc']:.3f}")
if REF['auc'] > 0.55:      # the ratio is meaningless when pooled AUC ~ chance
    print(f"skill above chance attributable to between-region contrast: "
          f"{100*(1-(aucs.mean()-0.5)/(REF['auc']-0.5)):.0f}%")
else:
    print(f"pooled AUC is at chance ({REF['auc']:.3f}); the between-region share "
          f"is undefined and is not reported")
_n_same = int((lobo_theta == REF['theta']).sum())
print(f"\nLeave-one-block-out threshold selection (Section 6.4's claim)")
print(f"  theta* trained on the other {n_scored_tiles-1} tiles, applied to each held-out tile:")
print(f"  identical to the in-sample {REF['theta']:.2f} on {_n_same}/{n_scored_tiles} tiles"
      f"   (range {lobo_theta.min():.2f}-{lobo_theta.max():.2f})")
_n_narrow = int((lobo_narrow == REF['theta']).sum())
print(f"  sensitivity, pool restricted to the other {n_scored_tiles-1} QUALIFYING tiles:"
      f" {_n_narrow}/{n_scored_tiles}"
      f"   (range {lobo_narrow.min():.2f}-{lobo_narrow.max():.2f})")
print(f"\nFor contrast, each tile's OWN optimum (a different question -- how far")
print(f"the local optima scatter, not whether the calibration transfers):")
print(f"  identical to {REF['theta']:.2f} on {int((own_theta == REF['theta']).sum())}"
      f"/{n_scored_tiles} tiles, median {np.median(own_theta):.2f},"
      f" range {own_theta.min():.2f}-{own_theta.max():.2f}")
ok("LOBO theta* transfers to all tiles", _n_same, n_scored_tiles, 0)

print()
ok("blocked tiles scored", n_scored_tiles, PUBLISHED['blocked_tiles'], 2)
ok("blocked tiles total", n_tiles_total, PUBLISHED['blocked_total'], 3)
ok("within-block AUC", float(aucs.mean()), PUBLISHED['blocked_auc'], 0.01)
ok("within-block AUC sd", float(aucs.std(ddof=1)), PUBLISHED['blocked_sd'], 0.015)

_a = P.alpha[SCORED]; _p = P.pga_g[SCORED]
print(f"\nslope vs modelled PGA over the scored domain: r = "
      f"{np.corrcoef(_a, _p)[0,1]:+.3f}")
print("  (a single geographic split cannot separate hazardous from safe terrain")
print("   on both variables at once, which is the concern this section tests)")

In [ ]:
# ============================================================================
#  §11  SATURATION SWEEP  ->  Section 5.8
#  m is held at 0.60 throughout. k_c is linear in m with
#  d k_c / d m = -(gamma_w/gamma) tan phi', so the consequences can be read off.
# ============================================================================
guard()

z5 = np.clip(P.z_raw, 0.1, 5.0); c10 = P.c + 10.0
print(f"{'m':>6}{'mean FoS':>11}{'unstable at 0.90':>19}{'theta*':>9}{'J':>9}{'AUC':>9}")
print("-" * 63)
SAT = {}
for mv in (0.0, 0.2, 0.4, 0.6, 0.8):
    mm = np.full(ref_shape, mv)
    f = np.full(ref_shape, np.nan)
    f[SCORED] = compute_FoS(c10[SCORED], P.phi[SCORED], P.alpha[SCORED],
                            mm[SCORED], P.gamma[SCORED], z5[SCORED], P.k_h[SCORED])
    s = SCORED & np.isfinite(f)
    p = np.sort(f[s & OBSERVED]); q = np.sort(f[s & ~OBSERVED])
    lt = np.searchsorted(q, p, 'left'); le = np.searchsorted(q, p, 'right')
    a = ((len(q)-le).sum() + 0.5*(le-lt).sum())/(len(p)*len(q))
    hr = np.searchsorted(p, THRESHOLDS, 'left')/len(p)
    fa = np.searchsorted(q, THRESHOLDS, 'left')/len(q)
    J = hr - fa; k = int(np.argmax(J))
    SAT[mv] = dict(fos_mean=float(np.nanmean(f[s])),
                   unstable=float(np.sum(s & (f < 0.90))/s.sum()),
                   theta=float(THRESHOLDS[k]), J=float(J[k]), auc=float(a))
    d = SAT[mv]
    print(f"{mv:>6.1f}{d['fos_mean']:>11.3f}{100*d['unstable']:>18.1f}%"
          f"{d['theta']:>9.2f}{d['J']:>9.3f}{d['auc']:>9.4f}")

_t = [SAT[v]['theta'] for v in (0.0, 0.2, 0.4, 0.6, 0.8)]
_slope = np.polyfit([0.0, 0.2, 0.4, 0.6, 0.8], _t, 1)[0]
print(f"\ntheta* migrates {_slope*0.1:+.3f} per 0.1 of m"
      f"   (d k_c/d m is constant, so this is linear by construction)")
print(f"J varies by {max(SAT[v]['J'] for v in SAT)-min(SAT[v]['J'] for v in SAT):.3f}"
      f" and AUC by {max(SAT[v]['auc'] for v in SAT)-min(SAT[v]['auc'] for v in SAT):.3f}"
      f" over m in [0, 0.8]")
print("-> saturation moves the CALIBRATION and leaves the DISCRIMINATION alone.")
_fac = 1 - P.m*GAMMA_W/P.gamma
print(f"\n(1 - m*gamma_w/gamma) over the scored domain at m=0.60: "
      f"{np.nanmean(_fac[SCORED]):.3f}")
print(f"  a dense soil at gamma=19 would retain "
      f"{1-0.6*GAMMA_W/19:.3f} of its friction")

In [ ]:
# ============================================================================
#  §12  k_c VARIANCE DECOMPOSITION   ->  the 71.5% soil-depth share
#
#  INDEPENDENT REIMPLEMENTATION. v9 computes this in a cell that reads the
#  global `scored`, and in a v9 Run All that mask had drifted by the end of the
#  notebook. This version runs on the guarded SCORED, so agreement with the
#  published 71.5% means the v9 figure was computed before the drift and is
#  safe; disagreement means it needs regenerating. Either way, COMPARE -- do
#  not silently replace.
#
#  First-order: Var(k_c) = sum_i (d k_c / d x_i)^2 Var(x_i), with
#      k_c = c/(gamma z cos^2 a) + (1 - m gamma_w/gamma) tan phi - tan a
# ============================================================================
guard()

SE_DIR = os.path.join(INPUT_DIR, "standard_error")
SE_FILES = {'gamma': 'buk_v1_standarderror.tif',
            'water': 'watercontent_v11standarderror.tif',
            'fines': 'siltclay_v1_standarderror.tif',
            'pi':    'PI_v1_standarderror.tif'}
REL_SE_Z = 0.30     # tab:kcvar states "assumed, 30% relative". The share is
                    # QUADRATIC in this, so it is the dominant assumption.
# Band-edge conversions for the threshold-assigned parameters. c' and m are cut
# from fines and water content at fixed thresholds, so neither has an ordinary
# derivative; some convention is needed. These were recovered by solving the
# paper's published shares, holding fixed the four terms that need no
# convention. See v10_extended_checks.py.
DC_DFINES = 0.3989      # kPa per pp of fines
DM_DWATER = 0.016766    # saturation ratio per pp of water content

def align_se(path):
    with rasterio.open(path) as src:
        a = src.read(1).astype(np.float64)
        if src.nodata is not None: a[a == src.nodata] = np.nan
        dst = np.full(ref_shape, np.nan)
        reproject(a, dst, src_transform=src.transform, src_crs=src.crs,
                  dst_transform=ref_transform, dst_crs=ref_crs,
                  resampling=Resampling.bilinear, src_nodata=np.nan, dst_nodata=np.nan)
    return dst

se = {}
for k, fn in SE_FILES.items():
    _path = os.path.join(SE_DIR, fn)
    if os.path.exists(_path):
        se[k] = align_se(_path)
        print(f"  {k:<6}: {fn}  coverage {100*np.isfinite(se[k][SCORED]).mean():.1f}% of scored")
    else:
        print(f"  {k:<6}: MISSING {fn}")

if len(se) < 4:
    print("\n!! not all standard-error rasters found; skipping the decomposition.")
    VAR_SHARES = None
else:
    a  = np.radians(P.alpha); ph = np.radians(P.phi)
    z5 = np.clip(P.z_raw, 0.1, 5.0); c10 = P.c + 10.0
    d  = P.gamma * z5 * np.cos(a)**2

    d_c     = 1.0/d
    d_z     = -c10/(P.gamma * z5**2 * np.cos(a)**2)
    d_m     = -(GAMMA_W/P.gamma) * np.tan(ph)
    d_phi   = (1 - P.m*GAMMA_W/P.gamma) / np.cos(ph)**2           # per radian
    d_gamma = -c10/(P.gamma**2 * z5 * np.cos(a)**2) + (P.m*GAMMA_W/P.gamma**2)*np.tan(ph)
    d_alpha = (2*c10*np.sin(a)/(P.gamma*z5*np.cos(a)**3)) - 1.0/np.cos(a)**2

    # input variances. c and m are threshold-assigned from fines and water, so
    # their variance is propagated through the band edges by a local-slope
    # approximation: a band edge at distance s SE away contributes the jump.
    var = {}
    var['z']     = (REL_SE_Z * z5)**2
    var['gamma'] = se['gamma']**2
    var['phi']   = (np.radians(0.1) * se['pi'])**2      # phi = 35 - 0.1*PI
    var['alpha'] = np.radians(2.0)**2                   # DEM slope, 2 deg
    var['c']     = (DC_DFINES * se['fines'])**2
    var['m']     = (DM_DWATER * se['water'])**2

    terms = {'z': d_z**2*var['z'], 'c(fines)': d_c**2*var['c'],
             'm(water)': d_m**2*var['m'], 'phi(PI)': d_phi**2*var['phi'],
             'gamma': d_gamma**2*var['gamma'], 'alpha': d_alpha**2*var['alpha']}
    tot = sum(terms.values())
    denom = np.nanmean(tot[SCORED])
    VAR_SHARES = {k: 100*np.nanmean(v[SCORED])/denom for k, v in terms.items()}

    print(f"\nFirst-order variance shares of k_c over the scored domain")
    print(f"  (relative SE on z assumed {REL_SE_Z:.0%}; the z share is quadratic in it)")
    for k, v in sorted(VAR_SHARES.items(), key=lambda x: -x[1]):
        print(f"    {k:<12} {v:>6.1f}%")
    print(f"    {'TOTAL':<12} {sum(VAR_SHARES.values()):>6.1f}%")
    print()
    ok("soil-depth share of Var(k_c)", VAR_SHARES['z']/100, PUBLISHED['depth_share'], 0.03)
    print("  ^ if this FAILS, the published 71.5% was computed on the drifted")
    print("    v9 mask, or on different variance assumptions. Check both before")
    print("    changing the manuscript -- REL_SE_Z above is the sensitive input.")

In [ ]:
# ============================================================================
#  §13  WRITE MAPS  ->  output_maps_v10/
# ============================================================================
guard()

# Built explicitly rather than inherited: the source profile carries tiling and
# dtype settings that need not be compatible with what is written here.
prof = dict(driver='GTiff', height=ref_shape[0], width=ref_shape[1], count=1,
            dtype='float32', crs=ref_crs, transform=ref_transform,
            nodata=NODATA, compress='lzw', tiled=True,
            blockxsize=256, blockysize=256)

def write_map(name, arr):
    out = np.where(np.isfinite(arr), arr, NODATA).astype('float32')
    _path = os.path.join(OUTPUT_DIR, name)
    with rasterio.open(_path, 'w', **prof) as dst:
        dst.write(out, 1)
    print(f"  wrote {name:<28} {os.path.getsize(_path)/1e6:>7.1f} MB")

masked = lambda a: np.where(SCORED, a, np.nan)
write_map("FoS_map.tif",             masked(REF['fos']))
write_map("FoS_static_map.tif",      masked(REF['fos_static']))
write_map("kc_map.tif",              masked(REF['kc']))
write_map("ac_map.tif",              masked(REF['kc']))          # a_c == k_c
write_map("sensitivity_FormB_map.tif", masked(FORMB_MAP))
write_map("failure_prediction.tif",  np.where(SCORED, (REF['fos'] < REF['theta']).astype(float), np.nan))
write_map("slope_deg.tif",           masked(P.alpha))
write_map("k_h_map.tif",             masked(P.k_h))
write_map("soil_depth_capped.tif",   masked(np.clip(P.z_raw, 0.1, 5.0)))

with open(os.path.join(OUTPUT_DIR, "run_manifest.json"), "w") as f:
    json.dump({
        'generated': time.strftime('%Y-%m-%d %H:%M:%S'),
        'grid_crs': str(ref_crs), 'lsi_crs': LSI_CRS,
        'reference_grid': REFERENCE_GRID, 'shape': list(ref_shape),
        'alpha_min': ALPHA_MIN, 'all_touched_observed': ALL_TOUCHED,
        'primary': [PRIMARY_C, PRIMARY_Z],
        'footprint_px': int(FOOTPRINT.sum()), 'scored_px': int(SCORED.sum()),
        'theta_star': REF['theta'], 'youden': REF['J_best'], 'auc': REF['auc'],
        'hit': REF['hit'], 'far': REF['far_best'],
        'checks_passed': sum(1 for _, g, _, _ in CHECKS if g),
        'checks_total': len(CHECKS),
    }, f, indent=2)
print(f"  wrote run_manifest.json")

In [ ]:
# ============================================================================
#  §14  FIGURES  ->  output_maps_v10/
# ============================================================================
guard()
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

def _show(ax, arr, title, cmap='viridis', norm=None, vmin=None, vmax=None):
    a = np.where(SCORED, arr, np.nan)
    im = ax.imshow(a, cmap=cmap, norm=norm, vmin=vmin, vmax=vmax, interpolation='nearest')
    ax.set_title(title, fontsize=9); ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.035, pad=0.02)

# ---- 6-panel overview ------------------------------------------------------
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
_show(axes[0,0], P.alpha, "Slope angle (deg)", 'terrain')
_show(axes[0,1], np.clip(P.z_raw, 0.1, 5.0), "Soil depth, 5 m cap (m)", 'YlOrBr')
_show(axes[0,2], P.k_h, "Seismic coefficient k_h", 'inferno')
_show(axes[1,0], REF['fos'], "Factor of Safety", 'RdYlGn',
      norm=TwoSlopeNorm(vmin=0, vcenter=REF['theta'], vmax=3))
_show(axes[1,1], REF['kc'], "Critical coefficient k_c", 'coolwarm_r',
      norm=TwoSlopeNorm(vmin=-0.5, vcenter=0, vmax=0.5))
_show(axes[1,2], sensitivity_FormB(REF['fos_static']), "Form B bound", 'magma')
fig.suptitle(f"Makilala v10 — primary configuration "
             f"({PRIMARY_C}, {PRIMARY_Z}), theta* = {REF['theta']:.2f}", fontsize=11)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "FoS_6panel.png"), dpi=150, bbox_inches='tight')
plt.close(fig); print("  wrote FoS_6panel.png")

# ---- ROC and Youden --------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))
for zn, lab in [('z_original','20 m cap'), ('z_cap_5m','5 m cap'), ('z_cap_3m','3 m cap')]:
    r = GRID[(PRIMARY_C, zn)]
    ax1.plot(r['far'], r['hr'], lw=1.6, label=f"{lab}  AUC={r['auc']:.3f}")
    j = int(np.argmax(r['J']))
    ax1.plot(r['far'][j], r['hr'][j], '*', ms=13)
    ax2.plot(THRESHOLDS, r['J'], lw=1.6, label=lab)
    ax2.axvline(r['theta'], ls=':', lw=1)
ax1.plot([0,1],[0,1],'k--',lw=0.8); ax1.set_xlabel("False alarm rate")
ax1.set_ylabel("Hit rate"); ax1.set_title("ROC"); ax1.legend(fontsize=8); ax1.grid(alpha=.3)
ax2.set_xlabel(r"FoS threshold $\theta$"); ax2.set_ylabel("Youden J")
ax2.set_title(r"$J(\theta)$ — a step function, flat near its maximum")
ax2.legend(fontsize=8); ax2.grid(alpha=.3)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "ROC_and_Youden.png"), dpi=150, bbox_inches='tight')
plt.close(fig); print("  wrote ROC_and_Youden.png")

# ---- validation panel ------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(17, 5.5))
_show(axes[0], REF['fos'], "FoS", 'RdYlGn',
      norm=TwoSlopeNorm(vmin=0, vcenter=REF['theta'], vmax=3))
pred = np.where(SCORED, (REF['fos'] < REF['theta']).astype(float), np.nan)
axes[1].imshow(pred, cmap='Reds', interpolation='nearest')
axes[1].set_title(f"Predicted unstable at theta*={REF['theta']:.2f}"
                  f"  ({FLAGGED_FRAC:.1f}% of domain)", fontsize=9)
axes[1].axis('off')
ov = np.full(ref_shape, np.nan); ov[SCORED] = 0
ov[SCORED & (REF['fos'] < REF['theta'])] = 1
ov[SCORED & OBSERVED] = 2
ov[SCORED & OBSERVED & (REF['fos'] < REF['theta'])] = 3
axes[2].imshow(ov, cmap='viridis', interpolation='nearest')
axes[2].set_title("0 stable  1 false alarm  2 missed  3 hit", fontsize=9); axes[2].axis('off')
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "validation_fig6.png"), dpi=150, bbox_inches='tight')
plt.close(fig); print("  wrote validation_fig6.png")

# ---- saturation / depth sensitivity ---------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
mv = sorted(SAT)
ax1.plot(mv, [SAT[v]['theta'] for v in mv], 'o-', label=r"$\theta^*$")
ax1.plot(mv, [SAT[v]['fos_mean'] for v in mv], 's-', label="mean FoS")
ax1.set_xlabel("saturation ratio m"); ax1.legend(); ax1.grid(alpha=.3)
ax1.set_title("Saturation moves the calibration")
ax2.plot(mv, [SAT[v]['J'] for v in mv], 'o-', label="Youden J")
ax2.plot(mv, [SAT[v]['auc'] for v in mv], 's-', label="AUC")
ax2.set_xlabel("saturation ratio m"); ax2.set_ylim(0.3, 0.85)
ax2.legend(); ax2.grid(alpha=.3); ax2.set_title("...and leaves the discrimination alone")
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "depth_sensitivity_6panel.png"), dpi=150, bbox_inches='tight')
plt.close(fig); print("  wrote depth_sensitivity_6panel.png")

In [ ]:
# ============================================================================
#  §15  MASTER SUMMARY   <-- PASTE THIS WHOLE BLOCK BACK
# ============================================================================
guard()

L = []
W = L.append
W("="*78)
W("MAKILALA FoS PIPELINE v10 — MASTER SUMMARY")
W(f"generated {time.strftime('%Y-%m-%d %H:%M:%S')}   ->  {OUTPUT_DIR}")
W("="*78)

W("\n-- DOMAIN (Table 2) --")
W(f"  full raster extent              {ref_shape[0]*ref_shape[1]:>12,}")
W(f"  municipal footprint             {int(FOOTPRINT.sum()):>12,}")
W(f"  + alpha >= {ALPHA_MIN} deg                {int((FOOTPRINT & (P.alpha>=ALPHA_MIN)).sum()):>12,}")
W(f"  + measured soil depth (SCORED)  {int(SCORED.sum()):>12,}")
W(f"  inventory: province / Makilala / scored   "
  f"{len(lsi):,} / {len(lsi_mak):,} / {n_poly_scored:,}")
W(f"  positive px: rasterised / scored         "
  f"{int(OBSERVED.sum()):,} / {int((SCORED & OBSERVED).sum()):,}")

W("\n-- PARAMETERS (Table 3) --")
W(f"  {'':<30}{'footprint min/max/mean':>34}{'scored min/max/mean':>34}")
for k, v in TABLE3.items():
    W(f"  {k:<30}{v[0]:>10.2f}{v[1]:>11.2f}{v[2]:>12.3f}"
      f"{v[3]:>11.2f}{v[4]:>11.2f}{v[5]:>12.3f}")

W("\n-- SWEEP (Table 1) --")
W(f"  {'cohesion':<12}{'z-cap':<12}{'c bar':>7}{'z bar':>7}{'FoS':>8}"
  f"{'theta*':>8}{'J':>8}{'hit':>8}{'FAR':>8}{'AUC':>9}")
for (cn, zn), r in GRID.items():
    W(f"  {cn:<12}{zn:<12}{r['c_mean']:>7.2f}{r['z_mean']:>7.2f}{r['fos_mean']:>8.3f}"
      f"{r['theta']:>8.2f}{r['J_best']:>8.3f}{100*r['hit']:>7.1f}%"
      f"{100*r['far_best']:>7.1f}%{r['auc']:>9.4f}")

W("\n-- PRIMARY OPERATING POINT (Table 4) --")
W(f"  theta* {REF['theta']:.2f}   J {REF['J_best']:.4f}   AUC {REF['auc']:.4f}")
W(f"  TP {N_TP:,}  FN {N_FN:,}  FP {N_FP:,}  TN {N_TN:,}")
W(f"  hit {REF['hit']:.4f}   FAR {REF['far_best']:.4f}   precision {prec:.4f}")
W(f"  flagged fraction of scored domain {FLAGGED_FRAC:.2f}%")

W("\n-- THRESHOLD OPTIMALITY --")
W(f"  exact sup J {J_EXACT:.6f} just above FoS {CAND_FOS[K_EXACT]:.6f}")
W(f"  grid  max J {REF['J_best']:.6f} at theta {REF['theta']:.2f}"
  f"   cost of gridding {J_EXACT-REF['J_best']:.6f}")
W(f"  J monotone on [0.05, 0.40]: {FLOOR_MONOTONE}   -> floor not binding")
W(f"  within 1e-3 of max: FoS {NEAR_OPT.min():.4f}-{NEAR_OPT.max():.4f}")

W("\n-- FORM B --")
W(f"  defined on {100*FORMB_DEFINED.sum()/SCORED.sum():.1f}% of scored, mean "
  f"{np.nanmean(FORMB_MAP[FORMB_DEFINED]):.4f}")

W("\n-- BLOCKED VALIDATION (Section 6.4) --")
W(f"  {n_scored_tiles} of {n_tiles_total} tiles scored")
W(f"  within-block AUC {aucs.mean():.3f} +/- {aucs.std(ddof=1):.3f}"
  f"  range {aucs.min():.3f}-{aucs.max():.3f}   pooled {REF['auc']:.3f}")
W(f"  LOBO theta* identical to in-sample on {int((lobo_theta == REF['theta']).sum())}"
  f"/{n_scored_tiles} tiles (pool = all scored px outside the tile)")
W(f"    narrower pool (other qualifying tiles only): "
  f"{int((lobo_narrow == REF['theta']).sum())}/{n_scored_tiles}")
W(f"    each tile's own optimum: "
  f"{int((own_theta == REF['theta']).sum())}/{n_scored_tiles}")

W("\n-- SATURATION SWEEP (Section 5.8) --")
for v in sorted(SAT):
    d = SAT[v]
    W(f"  m={v:.1f}  FoS {d['fos_mean']:.3f}  unstable {100*d['unstable']:>5.1f}%"
      f"  theta* {d['theta']:.2f}  J {d['J']:.3f}  AUC {d['auc']:.4f}")

if VAR_SHARES:
    W("\n-- k_c VARIANCE SHARES --")
    for kk, vv in sorted(VAR_SHARES.items(), key=lambda x: -x[1]):
        W(f"  {kk:<12} {vv:>6.1f}%")

W("\n" + "="*78)
_p = sum(1 for _, g, _, _ in CHECKS if g)
W(f"CHECKS: {_p} of {len(CHECKS)} reproduce the published value")
for lab, g, got, want in CHECKS:
    if not g:
        W(f"  FAIL  {lab:<44} got {got:>14}  published {want}")
if _p == len(CHECKS):
    W("All published values reproduced.")
W("="*78)

SUMMARY = "\n".join(L)
print(SUMMARY)
with open(os.path.join(OUTPUT_DIR, "master_summary.txt"), "w") as f:
    f.write(SUMMARY)
print(f"\n(also written to {os.path.join(OUTPUT_DIR, 'master_summary.txt')})")